# Structured Outputs & Practical Applications

This notebook covers:

1. **Structured Output Fundamentals** — Getting typed, validated data from LLMs
2. **Practical Application** — Building a Resume Analyzer & Job-Fit Scorer

**Docs**: [Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

## Setup

In [1]:
%pip install -qU langchain langchain-openai langgraph python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready!")

Environment ready!


---
## 1. Structured Output Fundamentals

Instead of getting free-form text from an LLM, **structured output** returns typed Python objects.

Use the `response_format` parameter on `create_agent()`.

### Why it matters:
- Parse-free: no regex or JSON parsing needed
- Type-safe: Pydantic validates the output
- Composable: feed structured results directly into downstream code

### Approach 1: TypedDict (simplest, returns dict)

In [16]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

agent = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": "Extract contact info: Hi I'm Jane Smith, reach me at jane@example.com or (555) 123-4567"
})

contact = result["structured_response"]
print(contact)
print(type(contact))  # dict

{'name': 'Jane Smith', 'email': 'jane@example.com', 'phone': '(555) 123-4567'}
<class 'dict'>


### Approach 2: Pydantic (recommended — validation + descriptions)

In [17]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

class MovieReview(BaseModel):
    """Structured analysis of a movie review."""
    title: str = Field(description="Movie title mentioned in the review")
    sentiment: Literal["positive", "negative", "mixed"] = Field(description="Overall sentiment")
    rating: int = Field(description="Rating from 1-10", ge=1, le=10)
    key_points: List[str] = Field(description="Main points from the review")
    recommended: bool = Field(description="Whether the reviewer recommends it")

review_agent = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=MovieReview,
)

result = review_agent.invoke({
    "messages": """Analyze this review:
    'Inception is a masterpiece. The layered dream sequences are mind-bending,
    the acting is superb, and Hans Zimmer's score is unforgettable. 
    My only gripe is the pacing in the first 20 minutes. Must watch!'"""
})

review = result["structured_response"]
print(f"Title: {review.title}")
print(f"Sentiment: {review.sentiment}")
print(f"Rating: {review.rating}/10")
print(f"Key Points: {review.key_points}")
print(f"Recommended: {review.recommended}")

Title: Inception
Sentiment: positive
Rating: 9/10
Key Points: ['Described as a masterpiece', 'Layered dream sequences are mind-bending', 'Acting is superb', "Hans Zimmer's score is unforgettable", 'Minor complaint about pacing in the first 20 minutes']
Recommended: True


### Strategies: ProviderStrategy vs ToolStrategy

| Strategy | How it works | Best for |
|----------|-------------|----------|
| `ProviderStrategy` | Uses the model's native structured output API | OpenAI, Anthropic, Gemini |
| `ToolStrategy` | Uses tool calling to extract structured data | Any model with tool support |
| Auto (pass schema directly) | LangChain picks the best strategy | Getting started |

When you pass a schema directly to `response_format`, LangChain auto-selects the best strategy.

In [18]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

# Explicit ProviderStrategy (native API — fastest)
agent_provider = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=ProviderStrategy(schema=MovieReview),
)

# Explicit ToolStrategy (universal fallback — works with tools too)
agent_tool = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=ToolStrategy(schema=MovieReview),
)

# Both produce the same structured output
r1 = agent_provider.invoke({"messages": "Review: The Matrix is amazing, 9/10, highly recommend!"})
r2 = agent_tool.invoke({"messages": "Review: The Matrix is amazing, 9/10, highly recommend!"})

print(f"Provider: {r1['structured_response'].title} — {r1['structured_response'].sentiment}")
print(f"Tool:     {r2['structured_response'].title} — {r2['structured_response'].sentiment}")

Provider: The Matrix — positive
Tool:     The Matrix — positive


---
## 2. Practical Application: Resume Analyzer & Job-Fit Scorer

Let's build something useful — an agent that:
1. Takes a **job description** and a **resume**
2. Extracts structured candidate profile data
3. Produces a **scored evaluation** with skills match, gaps, and a recommendation

This demonstrates **nested Pydantic models**, `Literal` enums, constrained fields, and real-world utility.

### Step 1: Define the Data Models

In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal


class SkillAssessment(BaseModel):
    """Assessment of a single skill."""
    skill_name: str = Field(description="Name of the skill")
    proficiency: Literal["expert", "proficient", "familiar", "none"] = Field(
        description="Candidate's proficiency level based on resume evidence"
    )
    evidence: str = Field(description="Brief evidence from the resume supporting this assessment")


class CandidateProfile(BaseModel):
    """Extracted candidate information."""
    name: str = Field(description="Candidate's full name")
    years_experience: int = Field(description="Total years of professional experience", ge=0)
    current_role: str = Field(description="Current or most recent job title")
    education: str = Field(description="Highest education level and field")
    top_skills: List[str] = Field(description="Top 5 technical skills from the resume")


class JobFitEvaluation(BaseModel):
    """Complete job-fit evaluation of a candidate."""
    candidate: CandidateProfile = Field(description="Extracted candidate profile")
    skill_assessments: List[SkillAssessment] = Field(
        description="Assessment of each required skill from the job description"
    )
    fit_score: int = Field(
        description="Overall fit score from 0-100", ge=0, le=100
    )
    strengths: List[str] = Field(description="Key strengths relative to the role")
    gaps: List[str] = Field(description="Missing skills or experience gaps")
    recommendation: Literal["strong_hire", "hire", "maybe", "no_hire"] = Field(
        description="Hiring recommendation"
    )
    summary: str = Field(description="2-3 sentence overall assessment")

print("Models defined! Schema has", len(JobFitEvaluation.model_fields), "top-level fields")
print("Nested models: CandidateProfile, SkillAssessment")

Models defined! Schema has 7 top-level fields
Nested models: CandidateProfile, SkillAssessment


### Step 2: Why Structured Output Matters — The Contrast

First, let's see what happens **without** structured output:

In [19]:
SAMPLE_JOB = """
Senior Python Developer — AI/ML Team

Requirements:
- 5+ years Python experience
- Experience with LangChain, LLM frameworks
- Strong SQL and database skills
- FastAPI or Django for API development
- Cloud experience (AWS or GCP)
- Nice to have: Kubernetes, MLOps
"""

SAMPLE_RESUME = """
Maria Santos
Senior Software Engineer | 7 years experience

EXPERIENCE:
- Senior Software Engineer at TechCorp (2022-present)
  Built ML pipelines using Python, LangChain, and FastAPI.
  Deployed models on AWS (ECS, Lambda, S3).
  Led a team of 3 engineers on the RAG search platform.

- Software Engineer at DataInc (2019-2022)
  Developed REST APIs with Django and PostgreSQL.
  Wrote complex SQL queries for analytics dashboard.
  Implemented CI/CD pipelines with GitHub Actions.

- Junior Developer at StartupXYZ (2017-2019)
  Full-stack development with Python and React.

EDUCATION:
- M.S. Computer Science, University of Lisbon (2017)

SKILLS: Python, LangChain, FastAPI, Django, PostgreSQL, AWS, Docker, Git, React
"""

# Unstructured approach — just text
unstructured_agent = create_agent(model="openai:gpt-5.4-mini", tools=[])

result = unstructured_agent.invoke({
    "messages": f"Evaluate this candidate for the role:\n\nJOB:\n{SAMPLE_JOB}\n\nRESUME:\n{SAMPLE_RESUME}"
})
print("--- Unstructured output (just text) ---")
print(result["messages"][-1].content[:500], "...")

--- Unstructured output (just text) ---
Overall: **Strong match / highly suitable**

### Why she fits well
- **Python experience:** 7 years, exceeds the 5+ year requirement.
- **LangChain / LLM frameworks:** Direct hands-on experience with **LangChain** and a **RAG search platform**.
- **SQL / databases:** Strong evidence from **PostgreSQL** and writing **complex SQL queries**.
- **API development:** Experience with both **FastAPI** and **Django**.
- **Cloud:** Solid **AWS** experience, including **ECS, Lambda, S3**.
- **Leadership:** ...


The text above is useful to read, but you can't programmatically access the score, recommendation, or skill gaps. Let's fix that.

### Step 3: Build the Structured Agent

In [24]:
resume_agent = create_agent(
    model="openai:gpt-5.4-mini",
    system_prompt="""You are an expert technical recruiter. Evaluate candidates against job descriptions.
    
    Be objective and evidence-based:
    - Only assess skills explicitly mentioned or clearly implied in the resume
    - Set proficiency to 'none' for skills not evidenced in the resume
    - Score fit_score based on coverage of required skills (not nice-to-haves)
    - Be specific in your evidence citations""",
    response_format=JobFitEvaluation,
)

SAMPLE_RESUME = """
John Wick
Senior Clown Engineer | 10 years experience

EXPERIENCE:
- Senior Clown Engineer at ClownCorp (2022-present)
  Built clown pipelines using clown tools
  Deployed clowns on circuses all over the place.
  Led a team of 3 clowns on the circus platform.
"""

result = resume_agent.invoke({
    "messages": f"Evaluate this candidate:\n\nJOB DESCRIPTION:\n{SAMPLE_JOB}\n\nRESUME:\n{SAMPLE_RESUME}"
})

evaluation = result["structured_response"]
print(type(evaluation))  # JobFitEvaluation

<class '__main__.JobFitEvaluation'>


### Step 4: Access the Structured Data

In [25]:
# Access nested fields programmatically
print(f"Candidate: {evaluation.candidate.name}")
print(f"Experience: {evaluation.candidate.years_experience} years")
print(f"Current Role: {evaluation.candidate.current_role}")
print(f"Education: {evaluation.candidate.education}")
print(f"Top Skills: {', '.join(evaluation.candidate.top_skills)}")
print()
print(f"FIT SCORE: {evaluation.fit_score}/100")
print(f"RECOMMENDATION: {evaluation.recommendation.upper()}")
print()
print(f"Summary: {evaluation.summary}")

Candidate: John Wick
Experience: 10 years
Current Role: Senior Clown Engineer
Education: 
Top Skills: Clown pipelines, Clown tools, Team leadership, Circus platform deployment, Operational delivery

FIT SCORE: 0/100
RECOMMENDATION: NO_HIRE

Summary: The resume provides no evidence of the core technical requirements for this Senior Python Developer role. While the candidate has substantial experience and some leadership background, the lack of demonstrated Python, AI/ML, database, API, and cloud skills makes this a very poor fit.


In [26]:
from IPython.display import display, HTML

rec_colors = {
    "strong_hire": ("#15803d", "#dcfce7"),
    "hire": ("#1d4ed8", "#dbeafe"),
    "maybe": ("#b45309", "#fef3c7"),
    "no_hire": ("#dc2626", "#fee2e2"),
}
rec_fg, rec_bg = rec_colors.get(evaluation.recommendation, ("#333", "#f3f4f6"))

skill_icons = {"expert": "★★★", "proficient": "★★·", "familiar": "★··", "none": "···"}
skill_colors = {"expert": "#15803d", "proficient": "#1d4ed8", "familiar": "#b45309", "none": "#9ca3af"}

skill_rows = ""
for s in evaluation.skill_assessments:
    c = skill_colors[s.proficiency]
    skill_rows += f"""
    <tr>
      <td style="padding:6px 12px;font-weight:600;white-space:nowrap;color:{c}">{skill_icons[s.proficiency]}</td>
      <td style="padding:6px 12px;font-weight:600">{s.skill_name}</td>
      <td style="padding:6px 12px"><span style="background:{c}22;color:{c};padding:2px 8px;border-radius:9999px;font-size:0.8em;font-weight:600">{s.proficiency}</span></td>
      <td style="padding:6px 12px;color:#6b7280;font-size:0.85em">{s.evidence}</td>
    </tr>"""

strengths_html = "".join(f"<li style='margin:4px 0'>{st}</li>" for st in evaluation.strengths)
gaps_html = "".join(f"<li style='margin:4px 0'>{g}</li>" for g in evaluation.gaps)

html = f"""
<div style="font-family:system-ui,-apple-system,sans-serif;max-width:780px;margin:12px 0;border:1px solid #e5e7eb;border-radius:12px;overflow:hidden;background:#fff">

  <div style="background:linear-gradient(135deg,#1e293b,#334155);padding:24px 28px;color:#fff;display:flex;justify-content:space-between;align-items:center">
    <div>
      <div style="font-size:1.5em;font-weight:700">{evaluation.candidate.name}</div>
      <div style="color:#94a3b8;margin-top:4px">{evaluation.candidate.current_role} &middot; {evaluation.candidate.years_experience} yrs exp &middot; {evaluation.candidate.education}</div>
    </div>
    <div style="text-align:center">
      <div style="font-size:2.4em;font-weight:800;line-height:1">{evaluation.fit_score}</div>
      <div style="font-size:0.75em;color:#94a3b8;letter-spacing:1px">FIT SCORE</div>
    </div>
  </div>

  <div style="padding:20px 28px">
    <div style="margin-bottom:12px">
      <span style="background:{rec_bg};color:{rec_fg};padding:4px 14px;border-radius:9999px;font-weight:700;font-size:0.9em;text-transform:uppercase;letter-spacing:.5px">{evaluation.recommendation.replace('_',' ')}</span>
      <span style="color:#6b7280;margin-left:12px;font-size:0.9em">{', '.join(evaluation.candidate.top_skills)}</span>
    </div>

    <p style="color:#374151;line-height:1.6;margin:14px 0">{evaluation.summary}</p>

    <details open style="margin-top:16px">
      <summary style="font-weight:700;font-size:1.05em;cursor:pointer;margin-bottom:8px">Skill Assessments</summary>
      <table style="width:100%;border-collapse:collapse;font-size:0.92em">
        <thead><tr style="border-bottom:2px solid #e5e7eb;text-align:left">
          <th style="padding:6px 12px"></th><th style="padding:6px 12px">Skill</th><th style="padding:6px 12px">Level</th><th style="padding:6px 12px">Evidence</th>
        </tr></thead>
        <tbody>{skill_rows}</tbody>
      </table>
    </details>

    <div style="display:flex;gap:24px;margin-top:18px">
      <div style="flex:1">
        <div style="font-weight:700;color:#15803d;margin-bottom:6px">Strengths</div>
        <ul style="margin:0;padding-left:20px;color:#374151;font-size:0.9em">{strengths_html}</ul>
      </div>
      <div style="flex:1">
        <div style="font-weight:700;color:#dc2626;margin-bottom:6px">Gaps</div>
        <ul style="margin:0;padding-left:20px;color:#374151;font-size:0.9em">{gaps_html}</ul>
      </div>
    </div>
  </div>

</div>
"""

display(HTML(html))

,Skill,Level,Evidence
···,Python experience,none,Resume does not mention Python or any Python-based projects.
···,LangChain / LLM frameworks,none,"No evidence of LangChain, LLMs, or AI/ML framework experience."
···,SQL and database skills,none,"No SQL, database, or data layer experience is mentioned."
···,FastAPI or Django,none,No API development frameworks are referenced.
···,Cloud experience (AWS or GCP),none,"Resume mentions 'deployed clowns on circuses all over the place' but does not specify AWS, GCP, or any cloud platform."


In [27]:
# Skill-by-skill breakdown
print("\nSkill Assessments:")
print("-" * 60)
for skill in evaluation.skill_assessments:
    icon = {"expert": "★★★", "proficient": "★★☆", "familiar": "★☆☆", "none": "☆☆☆"}[skill.proficiency]
    print(f"  {icon} {skill.skill_name:20s} [{skill.proficiency}]")
    print(f"      Evidence: {skill.evidence}")

print("\nStrengths:", evaluation.strengths)
print("Gaps:", evaluation.gaps)


Skill Assessments:
------------------------------------------------------------
  ☆☆☆ Python experience    [none]
      Evidence: Resume does not mention Python or any Python-based projects.
  ☆☆☆ LangChain / LLM frameworks [none]
      Evidence: No evidence of LangChain, LLMs, or AI/ML framework experience.
  ☆☆☆ SQL and database skills [none]
      Evidence: No SQL, database, or data layer experience is mentioned.
  ☆☆☆ FastAPI or Django    [none]
      Evidence: No API development frameworks are referenced.
  ☆☆☆ Cloud experience (AWS or GCP) [none]
      Evidence: Resume mentions 'deployed clowns on circuses all over the place' but does not specify AWS, GCP, or any cloud platform.

Strengths: ['10 years of professional experience', 'Leadership experience managing a team of 3']
Gaps: ['No evidence of Python experience', 'No evidence of AI/ML or LangChain/LLM framework work', 'No SQL/database experience shown', 'No FastAPI/Django API development experience shown', 'No AWS or GCP clo

### Step 5: Batch Evaluation — Compare Multiple Candidates

In [28]:
RESUME_2 = """
James Chen
ML Engineer | 3 years experience

EXPERIENCE:
- ML Engineer at AIStartup (2023-present)
  Fine-tuned LLMs using PyTorch and Hugging Face.
  Built evaluation pipelines for model performance.
  Some experience with LangChain for prototyping.

- Data Analyst at BigCo (2021-2023)
  Python scripting for data analysis.
  Basic SQL queries and Tableau dashboards.

EDUCATION:
- B.S. Statistics, UCLA (2021)

SKILLS: Python, PyTorch, Hugging Face, LangChain (basic), SQL, Tableau
"""

RESUME_3 = """
Aisha Patel
DevOps & Platform Engineer | 8 years experience

EXPERIENCE:
- Platform Lead at CloudNative Inc (2020-present)
  Designed Kubernetes clusters on GCP for ML workloads.
  Built MLOps pipelines with Kubeflow and Airflow.
  Python automation scripts for infrastructure.

- DevOps Engineer at ScaleUp (2018-2020)
  AWS infrastructure (EC2, RDS, EKS).
  CI/CD with Jenkins and Terraform.

- Systems Admin at LocalTech (2016-2018)
  Linux administration and shell scripting.

EDUCATION:
- B.S. Information Systems, Georgia Tech (2016)

SKILLS: Kubernetes, GCP, AWS, Python, Terraform, Docker, Airflow, MLOps, Linux
"""

# Evaluate all candidates
candidates = [
    ("Maria Santos", SAMPLE_RESUME),
    ("James Chen", RESUME_2),
    ("Aisha Patel", RESUME_3),
]

evaluations = []
for name, resume in candidates:
    r = resume_agent.invoke({
        "messages": f"Evaluate:\n\nJOB:\n{SAMPLE_JOB}\n\nRESUME:\n{resume}"
    })
    evaluations.append(r["structured_response"])
    print(f"Evaluated: {name}")

Evaluated: Maria Santos
Evaluated: James Chen
Evaluated: Aisha Patel


In [29]:
# Compare candidates — this is only possible with structured output!
print("\n" + "=" * 60)
print("CANDIDATE COMPARISON")
print("=" * 60)
print(f"{'Candidate':20s} {'Score':>6s}   {'Recommendation':15s}   Key Gap")
print("-" * 60)

# Sort by score
for ev in sorted(evaluations, key=lambda x: x.fit_score, reverse=True):
    gap = ev.gaps[0] if ev.gaps else "None"
    print(f"{ev.candidate.name:20s} {ev.fit_score:>5d}%   {ev.recommendation:15s}   {gap}")

print("\nTop candidate:", max(evaluations, key=lambda x: x.fit_score).candidate.name)


CANDIDATE COMPARISON
Candidate             Score   Recommendation    Key Gap
------------------------------------------------------------
Aisha Patel             48%   maybe             No evidence of LangChain or broader LLM framework experience.
James Chen              38%   no_hire           Does not meet the 5+ years Python requirement.
John Wick                0%   no_hire           No evidence of Python experience

Top candidate: Aisha Patel


### Try It Yourself!

Paste your own resume text below and evaluate it against any job description.

In [ ]:
# YOUR_JOB = """Paste a job description here"""
# YOUR_RESUME = """Paste your resume text here"""

# result = resume_agent.invoke({
#     "messages": f"Evaluate:\n\nJOB:\n{YOUR_JOB}\n\nRESUME:\n{YOUR_RESUME}"
# })
# ev = result["structured_response"]
# print(f"Score: {ev.fit_score}/100 — {ev.recommendation}")
# print(f"Summary: {ev.summary}")

---
## Summary

| Concept | API | When to Use |
|---------|-----|-------------|
| **Auto-selection** | `response_format=MyModel` | Getting started, simple cases |
| **ProviderStrategy** | `ProviderStrategy(schema=MyModel)` | Best performance, native support |
| **ToolStrategy** | `ToolStrategy(schema=MyModel)` | When you need tools + structured output |
| **TypedDict** | `class X(TypedDict)` | Quick prototyping (returns dict) |
| **Pydantic** | `class X(BaseModel)` | Production (validation + descriptions) |

**Key takeaway**: Structured output transforms LLMs from text generators into **data extraction engines**.

**Next**: Check out the [demo/](../demo/) directory for a deployable Chat-Over-Docs application.